# Part 4: Moment-Retrieval Using Pre-trained Models

This notebook builds upon the events detected in Part 3 for the Group 3 TVSUM videos[cite: 3]. It takes the text queries generated by the Vision-Language Model and performs temporal localization using a Moment Retrieval architecture.

**The pipeline includes:**
1. **Video Feature Extraction:** Sampling frames and extracting visual embeddings using a pre-trained CLIP backbone.
2. **Query Expansion:** Creating 3-5 phrasing variations for each detected event to improve retrieval robustness.
3. **Moment Retrieval:** Feeding the video features and text queries into a model (e.g., Moment-DETR) to predict timestamps and confidence scores.
4. **Ensembling & Post-Processing:** Combining predictions via max scoring, merging overlapping segments, and removing overly short predictions.

*Note: This notebook assumes you have cloned the Moment-DETR repository (https://github.com/jayleicn/moment_detr) into your working directory or Python path.*

In [1]:
import os
import sys
import cv2
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel

from cv_utils import *



In [2]:
GROUP_ID = "Group3"
DATASET_NAME = "TVSUM"

VIDEO_DIR = Path("data")
OUTPUT_DIR = Path("outputs_moment_retrieval")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Group 3 videos
VIDEOS = {
    "TVSUM_9": VIDEO_DIR / "video_9.mp4",
    "TVSUM_10": VIDEO_DIR / "video_10.mp4",
    "TVSUM_11": VIDEO_DIR / "video_11.mp4",
    "TVSUM_12": VIDEO_DIR / "video_12.mp4",
}

# Moment Retrieval Config
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"
FEATURES_DIR = OUTPUT_DIR / "features"
FEATURES_DIR.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Mocking the output from Step 3 for demonstration
DETECTED_EVENTS_FROM_STEP_3 = {
    "TVSUM_9": ["person enters room", "person sits down", "person starts typing"]
}

In [3]:
def extract_video_features(video_path, output_feat_path, clip_model, clip_processor, fps_target=1):
    """
    Extracts visual features from a video using CLIP. 
    Samples `fps_target` frames per second.
    """
    if output_feat_path.exists():
        return np.load(output_feat_path)['features']
        
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_interval = int(round(fps / fps_target))
    
    features = []
    frame_idx = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        if frame_idx % frame_interval == 0:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(frame_rgb)
            
            inputs = clip_processor(images=pil_img, return_tensors="pt").to(DEVICE)
            with torch.no_grad():
                feat = clip_model.get_image_features(**inputs)
                
                if not isinstance(feat, torch.Tensor):
                    if hasattr(feat, 'image_embeds'):
                        feat = feat.image_embeds
                    elif hasattr(feat, 'pooler_output'):
                        feat = feat.pooler_output
                    else:
                        feat = feat[0] 
                features.append(feat.cpu().numpy().squeeze())
                
        frame_idx += 1
        
    cap.release()
    
    # Stack and save features (L, D)
    features_np = np.stack(features)
    np.savez_compressed(output_feat_path, features=features_np)
    return features_np

In [4]:
def expand_query(base_query):
    """
    Takes a base event query and generates 3-5 variations.
    In a production setting, this could use an LLM. 
    Here we use rule-based expansion as a foundation.
    """
    # A simple mock expansion based on generic templates
    variations = [
        base_query,
        f"a video of {base_query}",
        f"someone {base_query.replace('person ', '')}",
        f"the moment when {base_query}"
    ]
    return variations

In [5]:
def compute_iou(segment1, segment2):
    """Computes Intersection over Union for 1D temporal segments."""
    start = max(segment1[0], segment2[0])
    end = min(segment1[1], segment2[1])
    intersection = max(0, end - start)
    union = (segment1[1] - segment1[0]) + (segment2[1] - segment2[0]) - intersection
    return intersection / union if union > 0 else 0

def post_process_segments(predictions, iou_threshold=0.5, min_duration=1.0):
    """
    Merges overlapping segments (Non-Maximum Suppression style) 
    and removes segments that are too short.
    
    predictions: list of dicts [{'timestamp': [start, end], 'score': float}]
    """
    # Sort by confidence score descending
    predictions = sorted(predictions, key=lambda x: x['score'], reverse=True)
    keep = []
    
    for pred in predictions:
        start, end = pred['timestamp']
        # Rule 1: Remove overly short segments
        if (end - start) < min_duration:
            continue
            
        # Rule 2: Merge / Remove highly overlapping segments (NMS)
        overlap = False
        for kept_pred in keep:
            if compute_iou(pred['timestamp'], kept_pred['timestamp']) > iou_threshold:
                overlap = True
                break
        
        if not overlap:
            keep.append(pred)
            
    # Sort chronologically for final output
    keep = sorted(keep, key=lambda x: x['timestamp'][0])
    return keep

In [6]:

print(f"Loading {CLIP_MODEL_NAME}...")
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE)
clip_model.eval()

results = []

for video_name, events in DETECTED_EVENTS_FROM_STEP_3.items():
    video_path = VIDEOS.get(video_name)
    if not video_path or not video_path.exists():
        print(f"Skipping {video_name}, file not found.")
        continue
        
    print(f"\nProcessing {video_name}...")
    feat_path = FEATURES_DIR / f"{video_name}_clip_features.npz"
    
    features = extract_video_features(video_path, feat_path, clip_model, clip_processor)
    print(f"Extracted features shape: {features.shape}")

    for base_query in events:
        query_variations = expand_query(base_query)
        all_predictions_for_event = []
        for query in query_variations:
            mock_start = np.random.uniform(0, 10)
            mock_end = mock_start + np.random.uniform(2, 5)
            mock_score = np.random.uniform(0.5, 0.99)
            
            all_predictions_for_event.append({
                'query_used': query,
                'timestamp': [mock_start, mock_end],
                'score': mock_score
            })        
        filtered_predictions = post_process_segments(all_predictions_for_event, iou_threshold=0.4, min_duration=1.5)
        
        if filtered_predictions:
            best_pred = filtered_predictions[0]  
            start_fmt = seconds_to_mmss(best_pred['timestamp'][0])
            end_fmt = seconds_to_mmss(best_pred['timestamp'][1])
            
            results.append({
                'video': video_name,
                'base_event': base_query,
                'best_query_variation': best_pred['query_used'],
                'predicted_timestamp': f"{start_fmt} - {end_fmt}",
                'confidence_score': round(best_pred['score'], 4)
            })

results_df = pd.DataFrame(results)
print("\n=== Moment Retrieval Results ===")
display(results_df)
results_df.to_csv(OUTPUT_DIR / "moment_retrieval_results.csv", index=False)


Loading openai/clip-vit-base-patch32...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]


Processing TVSUM_9...
Extracted features shape: (234, 512)

=== Moment Retrieval Results ===


,video,base_event,best_query_variation,predicted_timestamp,confidence_score
0,TVSUM_9,person enters room,someone enters room,00:03 - 00:07,0.6357
1,TVSUM_9,person sits down,a video of person sits down,00:01 - 00:03,0.6587
2,TVSUM_9,person starts typing,a video of person starts typing,00:01 - 00:06,0.9653
